In [ ]:
import argparse
import logging
import pathlib
import re
import math

import pandas as pd
import numpy as np
from Bio import SeqIO
from collections import Counter

import sgrna_subsample

import bioframe as bf
import bioframe.vis

In [ ]:
headpath = '/home/gutil/scratch/2023-11-15_Mismatch-CRISPRi-Essential-Genes/'
SEED_LENGTH = 9
SPACER_LENGTH = 20

## 1. Get list of genes with their positions

In [ ]:
# Get gene df
# TODO: Assuming there is only one record. To be extended to multiple records

headpath = '/home/gutil/scratch/2023-11-15_Mismatch-CRISPRi-Essential-Genes/'
path_genbank = headpath+'GCF_000009045.1_ASM904v1_genomic_Bsubtilis-168.gbff'
rec= SeqIO.read(path_genbank, "genbank")

# 'Upper' and 'Lower' strands of the genome
upper_seq = str(rec.seq)
lower_seq = str(rec.seq.reverse_complement())
GENOME_SIZE = len(upper_seq)

if not rec.features:
    raise ValueError("No features found in the genbank file")

# Initialize a list with a length of the number of genes
list_genes = []
for feature in rec.features:
    if feature.type == "CDS":
        list_genes.append([feature.qualifiers.get("locus_tag", ["NA"])[0],
                           feature.qualifiers.get("gene", ["NA"])[0],
                           feature.location.strand,
                           feature.location.start,
                           feature.location.end,
                        #    feature.extract(rec.seq)
                           ])

# Convert the list to a dataframe
df_genes = (pd.DataFrame(list_genes, columns=["locus_tag", "gene_name", "strand", "start", "end"])#, "sequence"])
            .assign(chrom = '0')
            .assign(matches_up_strand = lambda df_:df_['strand'] == 1)
            )

## Functions

In [ ]:
def find_seed_start_positions(query_seed, genome_seq):
    
    # I only need:
    # seed_start
    # matches_upper_seq (Not even, use an indicator for upper or lower just to flip the indices)
    # is_query_spacer
    seed_start = []
    for m in re.finditer(query_seed+r'.GG', genome_seq):
        seed_start.append(m.start() + 1) # +1 for 0-based index
    
    return seed_start
    
def find_all_targets(query_index, query_spacer, query_end_position):

        assert len(query_spacer) == 20
        query_seed = query_spacer[len(query_spacer) - SEED_LENGTH:]

        # Do it for the upper strand (+1)
        matches_up_strand = True
        genome_seq = upper_seq
        assert matches_up_strand and genome_seq == upper_seq

        seed_start_positions = find_seed_start_positions(query_seed, upper_seq)

        # print(seed_start_positions)

        # Create a dataframe with the seed start positions, the seed sequence, the spacer sequence, and the query end position
        df_seed_positions = pd.DataFrame(seed_start_positions, columns=["seed_start"])
        df_seed_positions = (df_seed_positions
                                .assign(query_index = query_index,
                                        query_spacer_sequence = query_spacer,
                                        seed_sequence = query_seed,
                                        query_end_position = query_end_position,
                                        matches_up_strand =  matches_up_strand # Strand that it matches (not the one that it binds to)
                                        )

                        .loc[:, ["query_index", "query_spacer_sequence", "seed_sequence", "query_end_position", "seed_start", "matches_up_strand"]]
                        
                        # Rearrange columns for better readability
        )

        # Do it for the lower strand (-1)
        matches_up_strand = False
        genome_seq = lower_seq
        assert (not matches_up_strand) and (genome_seq == lower_seq)

        seed_start_positions_rc = find_seed_start_positions(query_seed, lower_seq)

        # print(seed_start_positions_rc)

        # Create a dataframe with the seed start positions, the seed sequence, the spacer sequence, and the query end position
        df_seed_positions_lower = pd.DataFrame(seed_start_positions_rc, columns=["seed_start"])
        df_seed_positions_lower = (df_seed_positions_lower
                                .assign(query_index = query_index,
                                        query_spacer_sequence = query_spacer,
                                        seed_sequence = query_seed,
                                        query_end_position = query_end_position,
                                        matches_up_strand =  matches_up_strand, # Strand that it matches (not the one that it binds to)
                                        seed_start = lambda df_: GENOME_SIZE - df_["seed_start"] + 1 # Invert the indices (to the upper strand coordinates)
                                        )

                        .loc[:, ["query_index", "query_spacer_sequence", "seed_sequence", "query_end_position", "seed_start", "matches_up_strand"]]
                        
                        # Rearrange columns for better readability
        )
        # df_seed_positions_lower

        df_seed_positions = pd.concat([df_seed_positions, df_seed_positions_lower])
        return df_seed_positions

def find_all_targets_multiple_query(df):
    df = pd.concat(
        [find_all_targets(query_index, query_spacer, query_end_position) 
            for query_index, (query_spacer, query_end_position) in df.iterrows()
        ]
    )

    df = (df
    # Create column seed_end using np.where.
        .assign(seed_end = lambda df_: np.where(df_["matches_up_strand"],
                                                df_["seed_start"] + SEED_LENGTH - 1, 
                                                df_["seed_start"] - (SEED_LENGTH - 1)))
        .assign(spacer_start = lambda df_: np.where(df_["matches_up_strand"],
                                                    df_["seed_start"] - (SPACER_LENGTH - SEED_LENGTH),
                                                    df_["seed_start"] + (SPACER_LENGTH - SEED_LENGTH)))
        

        .assign(spacer_seq = lambda df_: df_.apply(lambda row: upper_seq[row["spacer_start"] - 1 : 
                                                                        row["seed_end"]]
                                                                if row["matches_up_strand"] 
                                                                else lower_seq[GENOME_SIZE - row['spacer_start'] : 
                                                                            GENOME_SIZE - (row['seed_end'] - 1)], axis=1))
        .reset_index(drop=True)
        
        .assign(n_nts_mismatched = lambda df_: df_.apply(lambda row: sum(c1 != c2 for c1, c2 in zip(row["query_spacer_sequence"], row["spacer_seq"])), axis=1))
        .astype({"n_nts_mismatched": 'uint8'})
        .assign(start = lambda df_: np.where(df_["matches_up_strand"],
                                            df_["spacer_start"], 
                                            df_["seed_end"]),
                end = lambda df_  : np.where(df_["matches_up_strand"],
                                            df_["seed_end"], 
                                            df_["spacer_start"]),
                chrom = '0')
        .astype({"start": 'int64', "end": 'int64'})
    )

    return df

## 2. Testing

In [ ]:
df_query_spacers_test = pd.DataFrame({'query_spacer': ['ATCTCTGTATCAATTGAAAT', 'ACCGCCAGACTGAATCACAT'],
                                      'query_end_position': [3218, 3270]})

df_hits = find_all_targets_multiple_query(df_query_spacers_test)

# Get the number of off-target hits
df_n_off_targets = (df_hits
                    .loc[lambda df_: df_['query_end_position'] != df_['seed_end']] # Filter out the on-targets
                    .groupby('query_index')
                    .size()
)

df_map = (bioframe.overlap(df1 = df_hits, df2 = df_genes, suffixes=('_seed', '_gene'), how = 'left')
        .drop(columns = ['chrom_gene', 'chrom_seed'])
        .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand = lambda df_: df_['matches_up_strand_seed'] != df_['matches_up_strand_gene'])
        .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets
        .sort_values(by = ['query_index_seed', 'binds_non_template_strand', 'n_nts_mismatched_seed'], ascending = [True, True, False])
        .reset_index(drop=True)
)

## 3. Controls

In [ ]:
# Do it for all the controls
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"
filename_controls = 'Table_S3_1-s2.0-S2405471220303665-mmc4.xlsx'
df_controls = (pd.read_excel(headpath+filename_controls,
                    sheet_name="relative fitness (bsu controls)")
                 # Remove row with nans
                 .rename(columns={'relative fitness (stddev)':'relative_fitness_std',
                                 'relative fitness (mean)':'relative_fitness_mean',
                                 'variant': 'mismatch_sgrna',
                                 'original': 'full_sgrna'}) # Rename column
                 .dropna(subset=['relative_fitness_mean'])
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.check_all_sites) == False] # Remove rows with restriction sites
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs).apply(sgrna_subsample.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Remove rows with BsaI and PvuI sites
                 # Remove rows for which 'relative fitness (stdev)' is less than 0.02
                #  .loc[lambda df_: df_['relative_fitness_std'] < 0.02]
                 # Filter rows whose relative_fitness_mean is between 1 + delta and 1 - delta
                 .loc[lambda df_: df_['relative_fitness_mean'].between(1-0.05, 1+0.5)] # TODO: Let delta be variable?
               #   .sample(n=100, replace=False) # Sample 100 controls
                 .assign(seed = lambda df_: df_['full_sgrna'].str[-9:],
                         category = 'control')
                 .astype({'category': 'category'})
                 .assign(seq_to_order = lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs))
                #  .loc[lambda df_: df_['seq_to_order'].apply(check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Check for BsaI and PvuI sites only
    )


In [ ]:
df_controls

In [ ]:
df_query_spacers = (df_controls
    .loc[:, ['mismatch_sgrna']]
    .rename(columns={'mismatch_sgrna': 'query_spacer'})
    .assign(query_end_position = None)
)
# Change name to query_spacer

In [ ]:
df_hits = find_all_targets_multiple_query(df_query_spacers)
# df_hits = pd.read_pickle(headpath+'df_hits_controls.pkl')

# Get the number of off-target hits
df_n_off_targets = (df_hits
                    .loc[lambda df_: df_['query_end_position'] != df_['seed_end']] # Filter out the on-targets
                    .groupby('query_index')
                    .size()
                    .to_frame()
                    # Remove the multi-level index
                    .reset_index()
                    .rename(columns={0: 'n_off_targets'})
                    .merge(df_query_spacers, left_on = 'query_index', right_index=True, how='right')
                    # Replace NaNs with 0 only in the column 'n_off_targets'
                    .fillna({'n_off_targets': 0})
                    .astype({'n_off_targets': 'uint32'})
)


In [ ]:

df_map = (bioframe.overlap(df1 = df_hits, df2 = df_genes, suffixes=('_seed', '_gene'), how = 'left')
        .drop(columns = ['chrom_gene', 'chrom_seed'])
        .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand = lambda df_: df_['matches_up_strand_seed'] != df_['matches_up_strand_gene'])
        .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets (NOT FOR CONTROLS)
        # This sorting should be WORST-FIRST so that we consider the worst off-targets when we filter queries
        .sort_values(by = ['query_index_seed', 'binds_non_template_strand', 'n_nts_mismatched_seed'], ascending = [True, False, True])
        .reset_index(drop=True)
)
df_map

In [ ]:
# Get the best possible off-target info
df_controls_final = (df_n_off_targets
    # Get the top hit for each query_index
    .merge(df_map.groupby('query_index_seed').head(1), left_on = 'query_index', right_on = 'query_index_seed', how='left')
    # .sort_values(by = ['query_index', 'n_off_targets', 'binds_non_template_strand', 'n_nts_mismatched_seed'], ascending = [True, False, True, False])
    # NOW SORT BY BEST-FIRST
    .sort_values(by = ['n_off_targets', 'binds_non_template_strand', 'n_nts_mismatched_seed'], ascending = [True, True, False])
    # .sort_values(by = ['binds_non_template_strand', 'n_off_targets', 'n_nts_mismatched_seed'], ascending = [True, True, False])
    # .loc[lambda df_: df_['n_off_targets'] >= 1]
    # Select only the following columns: query_index, n_off_targets, query_spacer, binds_non_template_strand, n_nts_mismatched_seed, gene_name_gene
    .loc[:, ['query_index', 'n_off_targets', 'query_spacer', 'binds_non_template_strand', 'n_nts_mismatched_seed', 'gene_name_gene']]
    .merge(df_controls, left_on = 'query_index', right_index=True, how='left')
    .loc[lambda df_:df_['relative_fitness_mean'] >= 0.94]
    .head(500)
    .reset_index(drop = True)
    # .loc[:,'binds_non_template_strand']
)
df_controls_final

In [ ]:
# TODO Verify that the off-targets are zero for the ones that do not show up in the map

## 4. Nonessential genes

# B. PREDICTION

In [ ]:
# Get prediction
DNA_BASES = ["A", "T", "C", "G"]

def get_lookup_table():
    lookup = pd.read_csv(headpath+'model_param_bsub.csv')
    features = lookup["feature"].to_list()
    weights = lookup["weight"].to_list()
    lookuptable = []
    for loc in range(20):
        for wt in range(len(DNA_BASES)):
            n_wt = DNA_BASES[wt]
            for mu in range(len(DNA_BASES)):
                n_mu = DNA_BASES[mu]
                if n_wt != n_mu:
                    P1 = float(weights[features.index(str(loc))]) + float(
                        weights[features.index(n_wt + n_mu)]
                    )
                    lookuptable.append((loc, n_wt, n_mu, P1))
                else:
                    P1 = 0
                    lookuptable.append((loc, n_wt, n_mu, P1))
    lookuptable = pd.DataFrame(lookuptable, columns=["loc", "wt", "mu", "P1"])

    loc = lookuptable["loc"].to_list()
    wt = lookuptable["wt"].to_list()
    mu = lookuptable["mu"].to_list()
    P1 = lookuptable["P1"].to_list()
    intercept = float(weights[features.index("intercept")])
    GC_weight = float(weights[features.index("GC_content")])
    return lookuptable, intercept, GC_weight

lookuptable, intercept, GC_weight = get_lookup_table()

DICT_BASES = {"A": 0, "T": 1, "C": 2, "G": 3}
def predict_efficacy(spacer_original, spacer_variant):
    # Find the index and the letter of the mismatch
    assert len(spacer_original) == len(spacer_variant) == 20
    mismatch = [(index, *substitution) 
                    for index, substitution in enumerate(zip(spacer_original, spacer_variant)) 
                        if substitution[0] != substitution[1]]

    # Get GC content of variant and originial
    GC_content = (spacer_variant.count("G") + spacer_variant.count("C")) / len(spacer_variant)
    GC_content_original = (spacer_original.count("G") + spacer_original.count("C")) / len(spacer_original)

    # For each entry in mismatch, find the position in the lookuptable
    return sum([lookuptable.iloc[
                mismatch[i][0]*16 + # 16 possible combinations of bp substitutions on each location
                DICT_BASES[mismatch[i][1]]*4 + # 4 basepairs: A, T, C, G
                DICT_BASES[mismatch[i][2]]
                ]['P1'] for i in range(len(mismatch))]) + intercept + GC_weight*GC_content_original 

In [ ]:
predict_efficacy('CCAGCAATAATGTTTAATTC', 'CCAGCAATAATCTTTAATTC')